# Упражнение: План агента: порядок подзадач, критический путь и бюджет токенов

## Задание

Агент-исследователь разбил задачу «подготовить отчёт» на 12 подзадач, которые образуют DAG
(ориентированный граф без циклов). Файл `input.csv`: `task_id`, `title`, `deps` (номера подзадач,
от которых зависит эта, через `;`; пусто — зависимостей нет), `tokens` (сколько токенов
стоит выполнение), `duration` (длительность в условных шагах). Файл `budget.json` —
`{"token_budget": N}`, общий бюджет токенов на весь план. Подзадачи с независимыми ветками
в принципе можно выполнять параллельно, но токены тратятся в любом случае.

Постройте план и разбор его выполнения:

1. **Порядок.** Топологический порядок (алгоритм Кана): на каждом шаге из всех готовых подзадач
   (все зависимости уже в порядке) берите подзадачу с **наименьшим `task_id`**. Номер в порядке — `order` (с 1).
2. **Расписание без ограничений по ресурсам.** `start` = максимум `finish` среди зависимостей (0, если их нет),
   `finish = start + duration`. Общая длительность плана T = максимальный `finish`.
3. **Запас.** Для каждой подзадачи найдите самое позднее время окончания `LF`, при котором план всё
   ещё укладывается в T: для подзадач без «детей» `LF = T`, иначе `LF` = минимум по детям
   (`LF(ребёнка) − duration(ребёнка)`). `slack = LF − finish`. Подзадачи с `slack = 0` лежат
   на критическом пути.
4. **Бюджет.** Проходите подзадачи в порядке `order` и копите потраченные токены. Статус подзадачи:
   - `blocked` — хотя бы одна зависимость не получила статус `done`;
   - иначе `skipped_budget`, если `spent + tokens > token_budget`;
   - иначе `done` (тогда `spent += tokens`).
   Дешёвая подзадача после пропущенной дорогой может поместиться в бюджет — алгоритм не останавливается.

## Подсказки

- Храните зависимости словарём `{task_id: [id, ...]}`, а для алгоритма Кана — ещё и словарь «детей» и счётчик
  входящих рёбер. Готовые подзадачи кладите в кучу `heapq`, тогда `heappop` сам даст наименьший номер.
- Прямой проход (start/finish) идёт по `order`, обратный проход (LF) — по `reversed(order)`:
  `lf[t] = min((lf[c] - duration[c] for c in children[t]), default=T)`.
- Статус считайте отдельным циклом по `order`: `blocked` проверяйте раньше бюджета, а `spent` увеличивайте
  только для `done`. В колонку `spent` пишите накопленную сумму **после** обработки подзадачи.

## Ожидаемый формат выходного файла

CSV с колонками `order, task_id, start, finish, slack, status, spent` — 12 строк в порядке `order`
(1…12). `status` — `done`, `skipped_budget` или `blocked`. Все значения — целые числа или строки.

## Проверка

Сохраните свой результат в `my_answer.csv` (в папке упражнения) и выполните из корня репозитория:

```bash
python tools/check_exercise.py exercises/agent_planning
```

Эталонное решение с пояснениями — `solution.py` (загляните в него только после своей попытки).
Верное решение портал отметит сам и откроет разбор (если он запущен через `python start.py`).

## Данные

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('input.csv')
print(df.shape)
df.head()

## Решение

In [ ]:
# Ваше решение. Итог запишите в DataFrame `result` — в формате, описанном в задании.

result = None

## Проверка

In [ ]:
# Проверка: сравнение result с эталоном (запускайте после своего решения).
# Сохраняет result в my_answer.csv, подсказывает вероятную причину ошибки, верное решение отмечает в портале.
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'tools' / 'check_exercise.py').exists())
sys.path.insert(0, str(root / 'tools'))
from check_exercise import check_result

check_result(result)

---
Эталонное решение с пояснениями — в файле `solution.py`. Загляните в него после своей попытки.